# Plan for today:
- PyTorch installation
- PyTorch basics
- Data loading and simple models in PyTorch (linear regression, images of digits)

# Install pytorch and torchvision
Installation instruction: https://pytorch.org/  

Different commands for GPU version -> all in PyTorch website

# PyTorch resources
- https://pytorch.org/tutorials/
- https://github.com/yunjey/pytorch-tutorial
- http://cs231n.stanford.edu/

# PyTorch basics

## Tensors



In [1]:
import torch

Basic datatype in PyTorch is a Tensor (syntax very similar to numpy array). https://pytorch.org/docs/stable/tensors.html  
**A `torch.Tensor` is a multi-dimensional matrix containing elements of a single data type.**

- A 0D tensor is scalar
- A 1D tensor is a vector 
- A 2D tensor is a matrix 
- A 3D tensor can be seen as a vector of identically size matrix
- A 4D tensor can be seen as a matrix of identically size matrices


In [2]:
data = torch.tensor([1, 2, 3])
print(data)
print(data.size())
print(data.dtype)

tensor([1, 2, 3])
torch.Size([3])
torch.int64


In [3]:
data = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
print(data)
print(data.size())

tensor([[1., 2., 3.],
        [4., 5., 6.]])
torch.Size([2, 3])


In [4]:
torch.zeros((3, 4))

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

In [5]:
torch.ones((3, 4))

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [6]:
# Random numbers: uniform [0, 1)
torch.rand(3, 5)

tensor([[0.1516, 0.4869, 0.3722, 0.1333, 0.6678],
        [0.2109, 0.0706, 0.5935, 0.2930, 0.8836],
        [0.9334, 0.8978, 0.4443, 0.4897, 0.1407]])

In [7]:
# More Operations
x = torch.tensor([[1,2,3], [2,4,6]])
# transpose
print(x.t())
# flatten
print(x.flatten())
# cropping
print(x[:, 1:3])

tensor([[1, 2],
        [2, 4],
        [3, 6]])
tensor([1, 2, 3, 2, 4, 6])
tensor([[2, 3],
        [4, 6]])


### For more operation see: https://pytorch.org/docs/stable/torch.html

## CPU vs GPU (CUDA)
Tensors can live on CPU or on GPU if torch is installed with Nvidia CUDA support. *CUDA tensors allow for much faster computation!*

In [8]:
torch.cuda.is_available()

True

To make code compatible with both GPU and CPU, you can set the device at the beginning of your code and always transfer your tensors/models to this device

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [10]:
print(device)

cuda


In [11]:
a = torch.tensor(5, device=device)
print(a)

tensor(5, device='cuda:0')


In [12]:
a = a.to(device)
print(a)

tensor(5, device='cuda:0')


In [13]:
a = a.to(torch.device('cpu'))
print(a)
print(a.device)

tensor(5)
cpu


## Computation

In [14]:
a = torch.tensor(5.0)
b = torch.tensor(2.0)
print(a, b)

tensor(5.) tensor(2.)


In [15]:
c = a + b
print(c)

tensor(7.)


In [16]:
c = a - b
print(c)

tensor(3.)


In [17]:
c = a * b
print(c)

tensor(10.)


In [18]:
c = a / b
print(c)

tensor(2.5000)


In [19]:
# Matrix computations
a = torch.rand(3, 4)
b = torch.rand(4, 1)
print(a)
print(b)

tensor([[0.5514, 0.0068, 0.2417, 0.0357],
        [0.7741, 0.6922, 0.4702, 0.2968],
        [0.7560, 0.2185, 0.3128, 0.8446]])
tensor([[0.6816],
        [0.0325],
        [0.7032],
        [0.9714]])


In [20]:
c = a.matmul(b)
print(c)

tensor([[0.5807],
        [1.1691],
        [1.5628]])


In [21]:
c = a @ b
print(c)

tensor([[0.5807],
        [1.1691],
        [1.5628]])


In [22]:
print(c + torch.rand(3, 1))

tensor([[1.5658],
        [1.5926],
        [1.8952]])


# PyTorch computational graph
Important property of PyTorch tensor operations - they create a **dynamic computational graph**.  
The tensor **c** knows that how it was created from **a** and **b**.  
Thanks to this, derivatives (gradients) can be computed for tensors.  
Example:  
<img src="tree-eval.png" alt="Drawing" style="width: 600px;"/>  
*Source: http://colah.github.io/posts/2015-08-Backprop/ - good blog post to understand backpropagation*

### Compute for a=2, b=1

In [23]:
# Create a and b
# Compute c, d, e
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
c = a + b
d = b + 1
e = c * d
print(a, b, c, d, e)

tensor(2., requires_grad=True) tensor(1., requires_grad=True) tensor(3., grad_fn=<AddBackward0>) tensor(2., grad_fn=<AddBackward0>) tensor(6., grad_fn=<MulBackward0>)


In [24]:
# grad_fn stores information about computation
print(c.grad_fn)
print(e.grad_fn)

### Now we can computer derivative of $e$ w.r.t. $a, b, c, d$
<img src="tree-eval-derivs.png" alt="Drawing" style="width: 600px;"/>  

$
\begin{equation} 
e = c * d
\end{equation}
$
$
\begin{equation} 
\frac {\partial e} {\partial c} = d = 2
\end{equation}
$
$
\begin{equation} 
\frac {\partial e} {\partial d} = c = 3
\end{equation}
$
$
\begin{equation} 
\frac {\partial e} {\partial a} = \frac {\partial e} {\partial c} \frac {\partial c} {\partial a} = 2
\end{equation}
$
$
\begin{equation} 
\frac {\partial e} {\partial b} = \frac {\partial e} {\partial c} \frac {\partial c} {\partial b} + \frac {\partial e} {\partial d} \frac {\partial d} {\partial b} = 2 * 1 + 3 * 1 = 5
\end{equation}
$


Compute gradients with `torch.autograd.grad` https://pytorch.org/docs/stable/autograd.html

In [25]:
# Compute gradients with torch.autograd.grad https://pytorch.org/docs/stable/autograd.html

In [26]:
grad_a, grad_b, grad_c, grad_d = torch.autograd.grad(e, [a, b, c, d])

In [27]:
print(grad_a, grad_b, grad_c, grad_d)

tensor(2.) tensor(5.) tensor(2.) tensor(3.)


In [28]:
# Another way for leaf tensors with `backward` method

In [29]:
# Create a and b
# Compute c, d, e
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
c = a + b
d = b + 1
e = c * d
print(a, b, c, d, e)

tensor(2., requires_grad=True) tensor(1., requires_grad=True) tensor(3., grad_fn=<AddBackward0>) tensor(2., grad_fn=<AddBackward0>) tensor(6., grad_fn=<MulBackward0>)


In [30]:
e.backward()

In [31]:
print(a.grad, b.grad, c.grad, d.grad)

tensor(2.) tensor(5.) None None


/tmp/ipykernel_1183/347313883.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  print(a.grad, b.grad, c.grad, d.grad)
